# Flashpoint Analytics

[Flashpoint Archive](https://flashpointarchive.org/?lang=en-US) is a web-game preservation project, made in 2018 in an effort to save as many games as possible from the then upcoming [Flash End-of-Life](https://www.adobe.com/products/flashplayer/end-of-life-alternative.html), while also making them playable for everyone. Today, it hosts more than 170 000 games and thousands of active users all around the world.

This notebook contains a descriptive statistical analysis about the games available in Flashpoint, with an emphasis on categorical data, such as the technology that was used to make them or the publisher who used to host them in the past.

The Flashpoint database, which keeps all the data that will be used in the analysis, can be found [here](https://web.archive.org/web/20230621043837/https://infinity.unstable.life/Flashpoint/Data/flashpoint.sqlite).

## Import modules

In [1]:
import os
import urllib.request
import sqlite3

These are the modules that we are going to use for our analysis. We are working with Python, but instead of relying on the *pandas* module like in the other notebook, we are going to write and run SQL queries directly to get the information we want.

## Retrieve data

In [2]:
try:
    os.mkdir("data")
except FileExistsError:
    pass

# download data from the source (it can take a while)

url = "https://web.archive.org/web/20230621043837id_/https://infinity.unstable.life/Flashpoint/Data/flashpoint.sqlite"
filename = "data/flashpoint.sqlite"
urllib.request.urlretrieve(url, filename)

# connect to the database and create a cursor

con = sqlite3.connect("data/flashpoint.sqlite")
cur = con.cursor()

We have created a connection with the database and a cursor which we will use to execute our queries and retrieve the results.

## Explore data 

Let's have a first look at our data.

In [3]:
res = cur.execute("PRAGMA table_info(game);").fetchall()
res

[(0, 'id', 'varchar', 1, None, 1),
 (1, 'parentGameId', 'varchar', 0, None, 0),
 (2, 'title', 'varchar', 1, None, 0),
 (3, 'alternateTitles', 'varchar', 1, None, 0),
 (4, 'series', 'varchar', 1, None, 0),
 (5, 'developer', 'varchar', 1, None, 0),
 (6, 'publisher', 'varchar', 1, None, 0),
 (7, 'dateAdded', 'datetime', 1, None, 0),
 (8, 'dateModified', 'datetime', 1, "datetime('now')", 0),
 (9, 'platform', 'varchar', 1, None, 0),
 (10, 'broken', 'boolean', 1, None, 0),
 (11, 'extreme', 'boolean', 1, None, 0),
 (12, 'playMode', 'varchar', 1, None, 0),
 (13, 'status', 'varchar', 1, None, 0),
 (14, 'notes', 'varchar', 1, None, 0),
 (15, 'source', 'varchar', 1, None, 0),
 (16, 'applicationPath', 'varchar', 1, None, 0),
 (17, 'launchCommand', 'varchar', 1, None, 0),
 (18, 'releaseDate', 'varchar', 1, None, 0),
 (19, 'version', 'varchar', 1, None, 0),
 (20, 'originalDescription', 'varchar', 1, None, 0),
 (21, 'language', 'varchar', 1, None, 0),
 (22, 'library', 'varchar', 1, None, 0),
 (23, 'o

We have used the PRAGMA command, which is exclusive to SQLite (it is one of the things that differentiate it from other SQL flavors such as MySQL, SQL Server, and so on).

There are 27 columns in the table: we can see the name of each field, its data type (mostly *varchar*, therefore characters and text), the possibility of the field being NULL (0) or not (1), its default value (which is None for almost all of them) and finally whether it is part of the primary key (1) or not (0). As we could expect, the *id* of the game, which is needed to uniquely identify it, serves as a primary key in the table structure.

We are going to need only some of these variables for our goal, so let's keep the relevant ones and look at the first rows of our dataset.

In [4]:
vars_to_keep = ["id", "title", "developer", "publisher", "platform", "releaseDate", "language", "library", "tagsStr"]
vars_to_keep = ", ".join(vars_to_keep)
vars_to_keep
res = cur.execute("SELECT %s FROM game LIMIT 3;" % (vars_to_keep)).fetchall()

for row in res:
    print(' | '.join(str(x) for x in row))

6db72888-6aa5-34c9-0ff3-ffe4cfe0fc61 | All Grown Up: Krazy Karts | Ezone | Nickelodeon | 3D Groove GX |  |  | arcade | Racing
3bba3af6-8e76-b2c8-b423-2d2d8bdfdd50 | Showdown: The Gunfighting Game | 3D Groove | 3D Groove | 3D Groove GX |  |  | arcade | Shooter
fb479276-2325-4dbb-bafd-64fcc8aeb684 | Hamsterball Bowling | Ezone | atv.Disney.go.com | 3D Groove GX |  |  | arcade | Arcade


We can already notice that there is some missing data, especially in the *Release Date* and *Language* fields.

## Analyze data 

### Developers and Publishers

We may be interested to know who are the most prolific developers and publishers. Let's find out by creating a frequency table for each variable and looking at the first ten entries.

In [5]:
top_developers = cur.execute("SELECT developer, count(*) FROM game GROUP BY developer ORDER BY count(*) DESC LIMIT 10").fetchall()
top_developers

[('', 46151),
 ('123Bee', 2730),
 ('Games2Rule', 2441),
 ('Games2Jolly.com', 1915),
 ('Selfdefiant', 1603),
 ('WowEscape.com', 1436),
 ('Top10NewGames', 1144),
 ('PalmarianFire', 1059),
 ('Ena Game Studio', 1031),
 ('Neopets', 957)]

The first row is blank because of the issue we noticed earlier: some games do not have a developer value associated to them in the database. Let's filter out those entries.

In [6]:
top_developers = cur.execute("SELECT developer, count(*) FROM game WHERE developer != '' GROUP BY developer ORDER BY count(*) DESC LIMIT 10").fetchall()
top_developers

[('123Bee', 2730),
 ('Games2Rule', 2441),
 ('Games2Jolly.com', 1915),
 ('Selfdefiant', 1603),
 ('WowEscape.com', 1436),
 ('Top10NewGames', 1144),
 ('PalmarianFire', 1059),
 ('Ena Game Studio', 1031),
 ('Neopets', 957),
 ('Mirchi Games', 812)]

These are the most represented developers in the database. As a Flash games fan may notice at first glance, most of them are specialised in escape games; therefore, we can suppose that it should be a very popular genre: we will dig into this later.

Before moving on, Neopets deserves a special mention for how it managed to build a passionate community still active after over 25 years since its release in 1999.

In [7]:
top_publishers = cur.execute("SELECT publisher, count(*) FROM game WHERE publisher != '' GROUP BY publisher ORDER BY count(*) DESC LIMIT 10").fetchall()
top_publishers

[('DeviantArt', 7655),
 ('Newgrounds', 6550),
 ('Disney', 2428),
 ('Nickelodeon', 1659),
 ('Armor Games', 1437),
 ('GameMonetize', 1374),
 ("Eka's Portal", 1261),
 ('Cartoon Network', 1243),
 ('Kongregate', 1219),
 ('Melting-Mindz', 1214)]

Among the publishers, we can see some very renowned names, at least in the gaming community, like *Newgrounds*, *Armor Games* and *Kongregate*. There is also a considerable amount of licensed games published by TV broadcasters, such as *Disney*, *Nickelodeon* and *Cartoon Network*, supposedly to promote their shows.

The main publishing platform is DeviantArt, which, despite being known mostly as a place for artists to store their drawings, also offers the possibility to share Flash content.

### Release Dates and Platforms

Flash games started to appear towards the end of the twentieth century and became popular in the next decade, before slowly fading out in favour of mobile games.

Let's observe the release dates we have got here, keeping in mind that they are not specified for all games.

In [8]:
older_dates = cur.execute("SELECT title, releaseDate, platform, library FROM game WHERE releaseDate != '' ORDER BY releaseDate LIMIT 10").fetchall()
older_dates

[('TankTrouble', '16-12-2007', 'Flash', 'arcade'),
 ('Blastar', '1984', 'HTML5', 'arcade'),
 ('Idle Johnny', '1993', 'Shockwave', 'theatre'),
 ('QP-Shot 1000', '1994', 'Shockwave', 'arcade'),
 ('The Health Checkup', '1994', 'Shockwave', 'theatre'),
 ('ZZZ...I want to sleep', '1994', 'Shockwave', 'theatre'),
 ('Dangerous Two', '1994', 'Shockwave', 'theatre'),
 ('Virtual Banana Original', '1994-02', 'VRML', 'arcade'),
 ('Virtual University of Auckland', '1994-02', 'Hyper-G', 'arcade'),
 ('Clock', '1994-11-17', 'Hyper-G', 'arcade')]

In [9]:
newer_dates = cur.execute("SELECT title, releaseDate, platform, library FROM game WHERE releaseDate != '' ORDER BY releaseDate DESC LIMIT 10").fetchall()
newer_dates

[('Havok Xtra RC Car Demo', '7/1/2002', 'Shockwave', 'arcade'),
 ("(Gift) Luna's Christmas gift", '2917-12-26', 'Flash', 'theatre'),
 ('Ray Cast Car', '27/06/2001', 'Shockwave', 'arcade'),
 ('Havok Xtra Marble Demo', '21/08/2001', 'Shockwave', 'arcade'),
 ('GWL Hayley Footjob (Commission)', '2080-10-05', 'Flash', 'arcade'),
 ('T-Mobile Tuesdays: Win $2,300 for 2023!', '2022-12-27', 'HTML5', 'arcade'),
 ('Awesome Game', '2022-12-23', 'HTML5', 'arcade'),
 ('Christmas in Vienna', '2022-12-23', 'HTML5', 'arcade'),
 ('Chrysler Building', '2022-12-23', 'HTML5', 'arcade'),
 ('Defender of Ukraine', '2022-12-20', 'HTML5', 'arcade')]

There seems to be a problem with the data: entries should follow the "YYYY-MM-DD" date format as per Flashpoint guidelines, but some games come with a different one; in addition, if the exact day or month of release is unknown, specifying the year only is also allowed.

Ignoring the problematic data, we see that the oldest game in the list is *Blastar*, which was released in 1984. Actually, the game present in Flashpoint is a HTML5 version, which was developed and released much more recently.

Moving on, starting from 1993 we recognize some old technologies, such as *Shochwave*, *VRML* and *Hyper-G*.

We can actually distinguish between proper games and animations by looking at the *library* column: the former are labeled with *arcade*, the latter with *theatre* values. Thus, the oldest animation featured is *Idle Johnny* from 1993, while the first "true" game (not counting *Blastar*) could be either *QP-Shot 1000* (which came out some time in 1994), or *Virtual Banana Original* and *Virtual University of Auckland*, both from February 1994.

On the other side, looking at more recent games, we find out that nowadays *HTML5* is the standard technology to make browser games.

For the sake of completeness, let's restrict our search to *Flash*-only games.

In [10]:
older_flash = cur.execute("SELECT title, releaseDate, platform, library FROM game WHERE releaseDate != '' and platform = 'Flash' ORDER BY releaseDate LIMIT 10").fetchall()
older_flash

[('TankTrouble', '16-12-2007', 'Flash', 'arcade'),
 ('Claus.com', '1995', 'Flash', 'arcade'),
 ("2 Design's Navigational Demo", '1996', 'Flash', 'arcade'),
 ('CHAOS Website', '1996', 'Flash', 'arcade'),
 ('FutureWave Software, Inc. Website', '1996', 'Flash', 'arcade'),
 ('Good Music Company Website', '1996', 'Flash', 'arcade'),
 ('The Silicon Slip', '1996', 'Flash', 'arcade'),
 ('Zygomedia Website', '1996', 'Flash', 'arcade'),
 ('First MouseOver Button', '1996', 'Flash', 'arcade'),
 ('Simple, Tasty Buttons', '1996', 'Flash', 'arcade')]

The first *Flash* game is *Claus.com* from 1995. We notice from the titles that most of these are actually websites built in *Flash* and not individual games or animations.

To take an overall view, let's compare the various platforms by games count, considering the top five.

In [11]:
top_platforms = cur.execute("SELECT platform, count(*) FROM game GROUP BY platform ORDER BY count(*) DESC LIMIT 5;").fetchall()
top_platforms

[('Flash', 141170),
 ('HTML5', 21963),
 ('Shockwave', 6232),
 ('Unity', 2107),
 ('Java', 1465)]

*Flash* is clearly the winner, followed by a rising *HTML5* and its old companion *Shockwave*, with *Unity* and *Java* as outsiders.

## Most common languages

Let's move on to another topic: *Flashpoint* allows non-English content as well, and it can be interesting to know which countries have contributed the most to the world of web games aside from the anglophone ones.

In [12]:
top_languages = cur.execute("SELECT language, count(*) FROM game WHERE language != '' and language NOT LIKE '%en%' GROUP by language ORDER BY count(*) DESC LIMIT 10").fetchall()
top_languages

[('ja', 5928),
 ('ko', 2196),
 ('pt', 1793),
 ('zh', 1141),
 ('pl', 1042),
 ('es', 773),
 ('fr', 556),
 ('de', 489),
 ('ru', 449),
 ('fi', 292)]

We can see a strong presence of Asian content, with Japanese, Korean and Chinese among the top ten languages. The rest of the list is completed by several European countries.

## Most popular genres

Let's now focus on game genres, featured on the *tagsStr* column, to discover the most common ones.

In [13]:
top_genres = cur.execute("SELECT tagsStr, count(*) FROM game WHERE tagsStr != '' GROUP by tagsStr ORDER BY count(*) DESC LIMIT 10").fetchall()
top_genres

[('Adventure; Escape the Room', 11718),
 ('Dress Up', 5334),
 ('Arcade', 5245),
 ('Puzzle', 4649),
 ('Escape the Room; Adventure', 4458),
 ('Comedy', 1902),
 ('Other', 1767),
 ('Toy', 1716),
 ('Find; Puzzle', 1699),
 ('Platformer', 1658)]

The big three genres are *Arcade*, *Puzzle* and *Adventure* and honestly it's kind of odd to see *Action* at such a low position. Conversely, as we expected from our previous analysis on developers, *Escape the Room* is fairly popular, along with *Dress Up* and *Simulation* games.

In [14]:
con.close()

## Most played games

As a final insight, let's find out which are the most played games among the *Flashpoint* users: to do this, we are going to use some official statistics from the platform itself. Visit [this webpage](https://web.archive.org/web/20230622070602/https://flashpoint-analytics.unstable.life/), scroll down to the corresponding section and download the data in *.csv* format.

In [15]:
import pandas

con = sqlite3.connect("data/most_played.sqlite")

df = pandas.read_csv("data/most_played.csv")
df.to_sql("most_played", con, if_exists='append', index=False)

cur = con.cursor()

most_played = cur.execute("SELECT * FROM most_played").fetchall()

for row in most_played[:10]:
    print(row)
con.close()


('83e1b5e7-4282-4bbd-868e-dcfa965e4abf', 56574)
('a94d865c-cb38-4d31-96f3-dda26502c4a3', 12194)
('617ca7f3-1cff-3f0c-5b53-07498b3b28d8', 9224)
('8d09fc0d-6f25-4be6-b396-8fcaddad4e5e', 8667)
('b0ce771e-7c02-4317-8528-ba48139e2688', 8627)
('fdee4800-b5c9-49e0-b19e-22f2b0ccab68', 8349)
('1e903a30-5c37-15bb-8e5e-6fea5a8103f2', 8042)
('b9a8dbb9-0cd7-434b-b226-13dc9dd07b49', 7724)
('07921a2f-26fd-4364-9671-ee0c8d256ec1', 7605)
('92ba2d91-e041-2bd4-49ea-21758df711ff', 7151)


The file contains the *id* for the most 40 played games, along with a play count. Let's use the identifiers to find the titles of these games and other info by combining the main dataframe and the new data.

In [16]:
ids = []

for row in most_played:
    id = row[0]
    ids.append(id)

con = sqlite3.connect("data/flashpoint.sqlite")
cur = con.cursor()

placeholders = ",".join(["?"] * len(ids)) 
query = f"SELECT title FROM game WHERE id IN ({placeholders})"
titles = cur.execute(query, ids).fetchall()
titles


[('Bloons TD 5',),
 ('Swords and Sandals 2',),
 ('Madness: Project Nexus',),
 ('Super Mario 63',),
 ('Papa Louie 2: When Burgers Attack!',),
 ('Road of the Dead',),
 ("Papa's Scooperia",),
 ('The Impossible Quiz',),
 ('Commando 2',),
 ("Papa's Pizzeria",),
 ('Bowman',),
 ("Papa's Pastaria",),
 ('Flappy Bird For Dinner',),
 ('Cactus McCoy',),
 ('Epic Battle Fantasy 5',),
 ('Swords and Souls',),
 ("Papa's Cheeseria",),
 ('Strike Force Heroes 2',),
 ('Poptropica',),
 ('Strike Force Heroes 3',),
 ('The Last Stand: Union City',),
 ("Papa's Hot Doggeria",),
 ("Papa's Freezeria",),
 ("Papa's Donuteria",),
 ("Papa's Pancakeria",),
 ("Papa's Bakeria",),
 ('Jacksmith',),
 ('Papa Louie 3: When Sundaes Attack!',),
 ('Super Mario Bros. Crossover',),
 ('Portal: The Flash Version',),
 ("Papa's Sushiria",),
 ("Papa's Wingeria",),
 ('Age of War',),
 ('Electricman 2 - The Tournament of Voltagen',),
 ('Plants vs Zombies (Web Version)',),
 ('Ben 10: Battle Ready',),
 ("Papa's Cupcakeria",),
 ('Super Smash

## Conclusion

This was a thorough analysis of the *Flashpoint* catalogue, which hopefully gives some insights about the world of web-based games and their significant relevance in the history of the Internet.

The effort to preserve this kind of content has generated amazing results, saving an astounding quantity of material which would have disappeared otherwise. Despite the concrete risk of a digital dark age, we should insist on preserving the stuff that we care about and keep it alive, not only for historical reasons, but also for the nostalgic value we associate with it.